# LLM Coding Notebook — Audience Analysis (Kenya)

**Norman Lear Center × Gates Foundation — Manfluencer Project**

Codes a stratified random sample of Kenya audience comments using `gpt-4o-mini` against the final codebook (`Codebooks/LLM Codebook/Nigeria Manfluencer Project - Final Coding Codebook.docx`). Same 13-theme vocabulary as Nigeria — the codebook is country-cross-cutting, validated against Kenya data via 20 keyword probes (none of the Kenya-specific candidates fired ≥1%).

## Inputs
- `Kenya/Audience Analysis/Kenya Audience Analysis Comments.xlsx` — 412 comments × 4 creators (Andrew Kibe, Onyango Otieno/Rixpoet, Eddy Kimani, Eric Amunga/Amerix).

## Sampling
Stratified 50 per creator → **200 comments**, balanced 50/50 progressive (Rixpoet + Eddy) vs regressive (Andrew + Amerix). Seed fixed at 42 for reproducibility.

## Output
Appends a new sheet **`Kenya - LLM Coding`** to the existing `Codebooks/LLM Codebook/LLM Coding - Audience Analysis.xlsx` (preserving Nigeria sheet).

## 0 — Setup

In [1]:
from __future__ import annotations
import asyncio, hashlib, json, os, re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm

ROOT = Path.cwd().resolve()
while ROOT.name != 'Gates-Manfluencer-Project' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert ROOT.name == 'Gates-Manfluencer-Project'

load_dotenv(ROOT / '.env')
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY missing'

KE_XLSX_IN = ROOT / 'Kenya' / 'Audience Analysis' / 'Kenya Audience Analysis Comments.xlsx'
OUT_DIR    = ROOT / 'Codebooks' / 'LLM Codebook'
OUT_XLSX   = OUT_DIR / 'LLM Coding - Audience Analysis.xlsx'
CACHE      = ROOT / 'temp' / 'llm_audience_coding_cache_kenya.parquet'
CACHE.parent.mkdir(parents=True, exist_ok=True)

MODEL = 'gpt-4o-mini'
SEED  = 42
N_PER_CREATOR = 50      # 50 × 4 = 200 total
CONCURRENCY = 8

ORIENTATION = {'Rixpoet':'progressive', 'Eddy':'progressive',
               'Andrew':'regressive', 'EricA':'regressive'}
CREATOR_FULL = {'Rixpoet':'Onyango Otieno (Rixpoet)', 'Eddy':'Eddy Kimani',
                'Andrew':'Andrew Kibe', 'EricA':'Eric Amunga (Amerix)'}

print(f'ROOT = {ROOT}')
print(f'model = {MODEL}, seed = {SEED}, n per creator = {N_PER_CREATOR} → total {N_PER_CREATOR*4}')

ROOT = /Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project
model = gpt-4o-mini, seed = 42, n per creator = 50 → total 200


## 1 — Load + stratified sample

In [2]:
import openpyxl
wb = openpyxl.load_workbook(KE_XLSX_IN, read_only=True)
all_dfs = []
for sn in wb.sheetnames:
    if sn == 'Summary Metrics': continue
    ws = wb[sn]
    rows = list(ws.iter_rows(values_only=True))
    hdr = rows[0]
    data = [dict(zip(hdr, r)) for r in rows[1:] if r and r[0]]
    sub = pd.DataFrame(data)
    all_dfs.append(sub)
df = pd.concat(all_dfs, ignore_index=True)
df['Comment'] = df['Comment'].astype(str)
df['orientation'] = df['Influencer'].map(ORIENTATION)
df['creator_full'] = df['Influencer'].map(CREATOR_FULL)
print(f'full Kenya audience: {len(df)} rows')
print(df['Influencer'].value_counts().to_string())

samples = []
for cr, sub in df.groupby('Influencer'):
    samples.append(sub.sample(n=N_PER_CREATOR, random_state=SEED))
sample = pd.concat(samples).reset_index(drop=True)
print(f'\nstratified sample: {len(sample)} rows')
print(sample.groupby(['orientation','Influencer']).size().to_string())

full Kenya audience: 412 rows
Influencer
EricA      140
Eddy       110
Rixpoet     92
Andrew      70

stratified sample: 200 rows
orientation  Influencer
progressive  Eddy          50
             Rixpoet       50
regressive   Andrew        50
             EricA         50


## 2 — Coding prompt (same as Nigeria — codebook is country-cross-cutting)

In [3]:
THEMES = [
    'Authority and Submission', 'Male Victimhood', 'Gender Grievance',
    'Sexual Morality', 'Relationship Tactics', 'Provider and Status',
    'Male Accountability', 'Egalitarian Partnership',
    'Gender-Based Violence and Consent', 'Trauma and Mental Health',
    'Self-Discipline', 'Marriage and Family', 'Faith and Moral Repair',
    'Unclear',
]
SENTIMENTS = ['Positive', 'Negative', 'Neutral', 'Unclear']
EMOTIONS = ['Joy', 'Happiness', 'Surprise', 'Anger', 'Fear', 'Contempt',
            'Sadness', 'Hope', 'Empathy', 'None of these']
TONES = ['Earnest', 'Sarcastic', 'Hostile', 'Humorous',
         'Empathetic', 'Authoritative', 'Detached']
NORMATIVE_ORIENTATIONS = ['Progressive', 'Regressive', 'Mixed', 'Unclear']
TARGETS = ['Men/boys', 'Women/girls', 'Feminists/modern women',
           'Children/family', 'Institutions/law/society', 'Creator/content',
           'Self/personal life', 'Mixed', 'Unclear']

PROMPT = f"""You are a senior research-grade content annotator coding Kenyan masculinity-related social-media AUDIENCE COMMENTS for the Norman Lear Center / Gates Foundation Manfluencer Project.

You are coding ONE comment at a time. Read the full comment carefully before assigning any code.

Treat each field as a defensible research judgment. Apply the codebook strictly. When in doubt, choose the more conservative option.

Do not invent labels. Use ONLY the exact options listed below.

Open-text fields must be specific and quote-grounded.

Note on Kenya context: Sheng/Swahili terms (mubaba, malaya, sponsor). Faith framing common (Christian + Islamic). Hustle/discipline language is part of the regressive register (Amerix-style).

CRITICAL CODING RULES (read first):

RULE 1 — Theme reflects what the COMMENTER advocates, not what the source post says. A commenter pushing back on misogyny is coded with the progressive theme they advance, NOT Gender Grievance or Male Victimhood.

RULE 2 — q4 INTERNALLY CONSISTENT with q5 and q6. If you assign a sentiment value to q5 or q6, q4 MUST be Yes. If q4 = No, q5 must be "Does not mention men/masculinity" and q6 must be "Does not mention women/femininity".

RULE 3 — q11 REQUIRED when q8 = Challenging. Empty q11 with Challenging q8 is a coding error.

RULE 4 — q16 HIGH BAR. Default = No. Yes ONLY when commenter explicitly says content "confirmed" or "proved" what they already believed. Mere agreement is NOT q16=Yes.

RULE 5 — q18 covers shared facts, claims, or personal info. "I had a husband who...", "in our culture we...", "research shows..." are q18=Yes.

RULE 6 — q20 covers correction or dispute. Yes when commenter explicitly says source/another commenter is wrong, incorrect, lying.

RULE 7 — q21 INTERNALLY CONSISTENT with q21a/c/e/g. If ANY of q21a/c/e/g is Yes, q21 MUST be Yes.

RULE 8 — Do NOT infer the commenter's identity from the source post.

PRIMARY + SECONDARY THEMES — pick ONE primary + up to 2 secondaries (no duplicates). If primary = "Unclear", secondaries empty.

1. Authority and Submission — explicit hierarchy.
2. Male Victimhood — men framed as harmed (only if COMMENTER claims this).
3. Gender Grievance — generalized distrust of women / feminists (COMMENTER must make the claim).
4. Sexual Morality — cheating, body count, body policing.
5. Relationship Tactics — tactical dating advice.
6. Provider and Status — money/status as masculine worth.
7. Male Accountability — men must change, hold men accountable. Use for comments defending women.
8. Egalitarian Partnership — mutual respect, equality, allyship.
9. Gender-Based Violence and Consent — rape, consent, abuse, false accusations.
10. Trauma and Mental Health — trauma, depression, vulnerability.
11. Self-Discipline — personal responsibility, growth.
12. Marriage and Family — marriage / family as the OBJECT of discussion.
13. Faith and Moral Repair — explicit faith / scripture / God / prayer / sin.
14. Unclear — uncodable.

masculinity_identity: "Yes" / "No"
normative_orientation: {' / '.join(NORMATIVE_ORIENTATIONS)} (code only what COMMENTER advocates)
target_of_claim: {' / '.join(TARGETS)}

sentiment: {' / '.join(SENTIMENTS)} (q1 MUST equal sentiment)
emotion: {' / '.join(EMOTIONS)} (q2 MUST equal emotion)
tone: {' / '.join(TONES)} (delivery, distinct from emotion)

HUMAN CODEBOOK Q1-Q21h — match human codebook exactly.
q1 = sentiment. q2 = emotion.
q3 emotional response multi-select: Feeling seen/understood, Feeling unseen/misunderstood, Feeling attacked, Feeling objectified, None of these
q4 mentions men/women/gender (Yes/No, RULE 2)
q5 sentiment toward men/masculinity (Pos/Neg/Neu/Unclear/Does not mention men/masculinity)
q6 sentiment toward women/femininity (Pos/Neg/Neu/Unclear/Does not mention women/femininity)
q7 main topic multi-select: The speaker/creator of the content/influencer/the content, Politics/social issues, Dating/relationships/marriage, Money/status, Fitness, Media/video games, Mental health/emotions, Gender roles/norms, Other
q7a if Other, one phrase, else empty
q8 stance (Supporting/Challenging/Neutral/Unclear)
q9 if Supporting, ONE sentence reason, else empty
q10 if Supporting, multi-select need; else ["None of these apply"]
q11 if Challenging, ONE sentence objection, REQUIRED, else empty
q12 personal experience (Yes/No)
q13 sexist language (Yes/No, high bar)
q14 new knowledge (Yes/No), q14a if yes
q15 attitude changed (Yes/No), q15a if yes
q16 opinion reinforced (No/Yes, RULE 4), q16a if yes
q17 calls to action (Yes/No), q17a if yes
q18 shares info (Yes/No, RULE 5), q18a if yes
q19 advocates (No/Yes), q19a if yes
q20 corrects (No/Yes, RULE 6), q20a if yes
q21 self-identifies (No/Yes, RULE 7)
q21a profession (No/Yes), q21b text else empty
q21c location (No/Yes), q21d text else empty
q21e race/ethnicity (No/Yes), q21f text else empty
q21g gender (No/Yes), q21h Female/Male/Non-binary/Other/Unclear else empty

OUTPUT JSON ONLY:
{{
  "primary_theme": "", "secondary_theme_1": "", "secondary_theme_2": "",
  "masculinity_identity": "", "normative_orientation": "", "target_of_claim": "",
  "sentiment": "", "emotion": "", "tone": "",
  "q1": "", "q2": "", "q3": [], "q4": "", "q5": "", "q6": "",
  "q7": [], "q7a": "",
  "q8": "", "q9": "", "q10": [], "q11": "",
  "q12": "", "q13": "",
  "q14": "", "q14a": "", "q15": "", "q15a": "", "q16": "", "q16a": "",
  "q17": "", "q17a": "", "q18": "", "q18a": "", "q19": "", "q19a": "",
  "q20": "", "q20a": "",
  "q21": "", "q21a": "", "q21b": "", "q21c": "", "q21d": "",
  "q21e": "", "q21f": "", "q21g": "", "q21h": ""
}}
"""
print(f'prompt length: {len(PROMPT):,} chars')


prompt length: 5,848 chars


## 3 — Run async (cached by SHA-1 of comment text)

In [4]:
def text_hash(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8')).hexdigest()[:16]

def load_cache():
    if CACHE.exists():
        c = pd.read_parquet(CACHE)
        return {r.text_hash: json.loads(r.result_json) for r in c.itertuples()}
    return {}

def save_cache(cache):
    rows = [{'text_hash': k, 'result_json': json.dumps(v, ensure_ascii=False)} for k, v in cache.items()]
    pd.DataFrame(rows).to_parquet(CACHE, index=False)

async def code_one(client, sem, text):
    async with sem:
        try:
            r = await client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': PROMPT},
                    {'role': 'user', 'content': text[:2000]},
                ],
                temperature=0,
                max_tokens=1200,
                response_format={'type': 'json_object'},
            )
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            return {'error': str(e)[:200]}

async def run_all():
    cache = load_cache()
    print(f'cache: {len(cache)} entries')
    todo = [(text_hash(t), t) for t in sample['Comment'] if text_hash(t) not in cache]
    print(f'rows to code: {len(todo)} / {len(sample)}')
    if not todo:
        return cache
    client = AsyncOpenAI()
    sem = asyncio.Semaphore(CONCURRENCY)
    tasks = [code_one(client, sem, t) for _, t in todo]
    results = await atqdm.gather(*tasks)
    for (h, _), res in zip(todo, results):
        cache[h] = res
    save_cache(cache)
    print(f'cached: {len(cache)} entries')
    return cache

cache = await run_all()

cache: 0 entries
rows to code: 200 / 200


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 1/200 [00:04<13:51,  4.18s/it]

  2%|▏         | 3/200 [00:04<03:46,  1.15s/it]

  2%|▎         | 5/200 [00:04<02:14,  1.45it/s]

...

 99%|█████████▉| 198/200 [03:01<00:01,  1.15it/s]

100%|█████████▉| 199/200 [03:02<00:00,  1.07it/s]

100%|██████████| 200/200 [03:03<00:00,  1.15it/s]

100%|██████████| 200/200 [03:03<00:00,  1.09it/s]

cached: 200 entries


## 4 — Validate + apply gating rules + assemble row data

In [5]:
with open(ROOT / 'temp' / 'human_audience_headers.json') as f:
    HUMAN_HDRS = json.load(f)
HDR_BY_KEY = {}
for h in HUMAN_HDRS:
    first = h.split('\n')[0].strip()
    if first.startswith('Q'):
        if '.' in first:
            key = first.split('.')[0].strip().lower().replace(' ', '')
        else:
            key = first.split()[0].lower()
        HDR_BY_KEY[key] = h

def hdr(q): return HDR_BY_KEY[q.lower()]

Q1_OPTS = ['Positive', 'Negative', 'Neutral', 'Unclear']
Q3_OPTS = ['Feeling seen/understood', 'Feeling unseen/misunderstood',
           'Feeling attacked', 'Feeling objectified', 'None of these']
Q5_OPTS = ['Positive','Negative','Neutral','Unclear','Does not mention men/masculinity']
Q6_OPTS = ['Positive','Negative','Neutral','Unclear','Does not mention women/femininity']
Q7_OPTS = ['The speaker/creator of the content/influencer/the content','Politics/social issues',
           'Dating/relationships/marriage','Money/status','Fitness','Media/video games',
           'Mental health/emotions','Gender roles/norms','Other']
Q8_OPTS = ['Supporting','Challenging','Neutral','Unclear']
Q10_OPTS = ['Entertainment/escapism','Information seeking','Connection/social interaction',
            'Self expression/identity construction','Status seeking','Documentation of events','None of these apply']
YN = ['Yes','No']
Q21H_OPTS = ['Female','Male','Non-binary','Other','Unclear']

def coerce_one(val, opts, default=''):
    if not isinstance(val, str): return default
    v = val.strip()
    if v in opts: return v
    for o in opts:
        if v.lower() == o.lower(): return o
    return default

def coerce_multi(val, opts):
    if not isinstance(val, list): return []
    seen = []
    for v in val:
        c = coerce_one(v, opts, default='')
        if c and c not in seen: seen.append(c)
    return seen

def sanitize_themes(p, s1, s2):
    pp  = coerce_one(p,  THEMES, 'Unclear')
    s1c = coerce_one(s1, THEMES, '')
    s2c = coerce_one(s2, THEMES, '')
    if pp == 'Unclear': return pp, '', ''
    if s1c == pp: s1c = ''
    if s2c == pp or s2c == s1c: s2c = ''
    return pp, s1c, s2c

fixlog = {'q4_consistency': 0, 'q21_parent_yes': 0, 'q11_required_missing': 0}

rows = []
for _, r in sample.iterrows():
    h = text_hash(r['Comment'])
    raw = cache.get(h, {})

    primary_theme, sec1, sec2 = sanitize_themes(
        raw.get('primary_theme'), raw.get('secondary_theme_1'), raw.get('secondary_theme_2'))

    masculinity_identity  = coerce_one(raw.get('masculinity_identity'),  YN, 'No')
    normative_orientation = coerce_one(raw.get('normative_orientation'), NORMATIVE_ORIENTATIONS, 'Unclear')
    target_of_claim       = coerce_one(raw.get('target_of_claim'),        TARGETS, 'Unclear')

    sentiment = coerce_one(raw.get('sentiment'), SENTIMENTS, 'Unclear')
    emotion   = coerce_one(raw.get('emotion'),   EMOTIONS,    'None of these')
    tone      = coerce_one(raw.get('tone'),      TONES,       'Detached')

    q1 = sentiment
    q2 = emotion
    q3  = coerce_multi(raw.get('q3'), Q3_OPTS); q3_str = '; '.join(q3) if q3 else 'None of these'

    q4 = coerce_one(raw.get('q4'), YN, 'No')
    q5 = coerce_one(raw.get('q5'), Q5_OPTS, 'Does not mention men/masculinity')
    q6 = coerce_one(raw.get('q6'), Q6_OPTS, 'Does not mention women/femininity')
    if q4 == 'No' and (q5 in ('Positive','Negative','Neutral','Unclear') or q6 in ('Positive','Negative','Neutral','Unclear')):
        q4 = 'Yes'; fixlog['q4_consistency'] += 1
    if q4 == 'No':
        q5 = 'Does not mention men/masculinity'; q6 = 'Does not mention women/femininity'

    q7 = coerce_multi(raw.get('q7'), Q7_OPTS); q7_str = '; '.join(q7) if q7 else 'Other'
    q7a = (raw.get('q7a') or '').strip()
    if 'Other' not in q7: q7a = ''

    q8 = coerce_one(raw.get('q8'), Q8_OPTS, 'Unclear')
    q9 = (raw.get('q9') or '').strip();   q9 = q9 if q8 == 'Supporting' else ''
    q10 = coerce_multi(raw.get('q10'), Q10_OPTS); q10_str = '; '.join(q10) if q10 else 'None of these apply'
    if q8 != 'Supporting': q10_str = 'None of these apply'
    q11 = (raw.get('q11') or '').strip(); q11 = q11 if q8 == 'Challenging' else ''
    if q8 == 'Challenging' and not q11:
        fixlog['q11_required_missing'] += 1

    q12 = coerce_one(raw.get('q12'), YN, 'No')
    q13 = coerce_one(raw.get('q13'), YN, 'No')
    q14  = coerce_one(raw.get('q14'),  YN, 'No'); q14a = (raw.get('q14a') or '').strip(); q14a = q14a if q14 == 'Yes' else ''
    q15  = coerce_one(raw.get('q15'),  YN, 'No'); q15a = (raw.get('q15a') or '').strip(); q15a = q15a if q15 == 'Yes' else ''
    q16  = coerce_one(raw.get('q16'),  YN, 'No'); q16a = (raw.get('q16a') or '').strip(); q16a = q16a if q16 == 'Yes' else ''
    q17  = coerce_one(raw.get('q17'),  YN, 'No'); q17a = (raw.get('q17a') or '').strip(); q17a = q17a if q17 == 'Yes' else ''
    q18  = coerce_one(raw.get('q18'),  YN, 'No'); q18a = (raw.get('q18a') or '').strip(); q18a = q18a if q18 == 'Yes' else ''
    q19  = coerce_one(raw.get('q19'),  YN, 'No'); q19a = (raw.get('q19a') or '').strip(); q19a = q19a if q19 == 'Yes' else ''
    q20  = coerce_one(raw.get('q20'),  YN, 'No'); q20a = (raw.get('q20a') or '').strip(); q20a = q20a if q20 == 'Yes' else ''

    q21a = coerce_one(raw.get('q21a'), YN, 'No')
    q21c = coerce_one(raw.get('q21c'), YN, 'No')
    q21e = coerce_one(raw.get('q21e'), YN, 'No')
    q21g = coerce_one(raw.get('q21g'), YN, 'No')
    q21_raw = coerce_one(raw.get('q21'), YN, 'No')
    if (q21a == 'Yes' or q21c == 'Yes' or q21e == 'Yes' or q21g == 'Yes') and q21_raw == 'No':
        q21 = 'Yes'; fixlog['q21_parent_yes'] += 1
    else:
        q21 = q21_raw
    if q21 == 'No':
        q21a = q21c = q21e = q21g = 'No'
        q21b = q21d = q21f = q21h = ''
    else:
        q21b = (raw.get('q21b') or '').strip(); q21b = q21b if q21a == 'Yes' else ''
        q21d = (raw.get('q21d') or '').strip(); q21d = q21d if q21c == 'Yes' else ''
        q21f = (raw.get('q21f') or '').strip(); q21f = q21f if q21e == 'Yes' else ''
        q21h = coerce_one(raw.get('q21h'), Q21H_OPTS, '')
        if q21g != 'Yes': q21h = ''

    rows.append({
        'Comment ID':                str(r['Comment ID']),
        'Commenter Post URL':        str(r.get('Source URL') or ''),
        "Influencer's OG Post URL":  str(r.get('Source URL') or ''),
        'Comment Text':              str(r['Comment']),
        'Primary Theme':             primary_theme,
        'Secondary Theme 1':         sec1,
        'Secondary Theme 2':         sec2,
        'Masculinity Identity':      masculinity_identity,
        'Normative Orientation':     normative_orientation,
        'Target of Claim':           target_of_claim,
        'Sentiment':                 sentiment,
        'Emotion Detection':         emotion,
        'Tone':                      tone,
        hdr('q1'):  q1, hdr('q2'): q2, hdr('q3'): q3_str, hdr('q4'): q4, hdr('q5'): q5, hdr('q6'): q6,
        hdr('q7'):  q7_str, hdr('q7a'): q7a,
        hdr('q8'):  q8, hdr('q9'): q9, hdr('q10'): q10_str, hdr('q11'): q11,
        hdr('q12'): q12, hdr('q13'): q13,
        hdr('q14'): q14, hdr('q14a'): q14a, hdr('q15'): q15, hdr('q15a'): q15a,
        hdr('q16'): q16, hdr('q16a'): q16a,
        hdr('q17'): q17, hdr('q17a'): q17a, hdr('q18'): q18, hdr('q18a'): q18a,
        hdr('q19'): q19, hdr('q19a'): q19a, hdr('q20'): q20, hdr('q20a'): q20a,
        hdr('q21'): q21, hdr('q21a'): q21a, hdr('q21b'): q21b,
        hdr('q21c'): q21c, hdr('q21d'): q21d, hdr('q21e'): q21e, hdr('q21f'): q21f,
        hdr('q21g'): q21g, hdr('q21h'): q21h,
        'creator':     r['creator_full'],
        'orientation': r['orientation'],
    })
out = pd.DataFrame(rows)
print(f'coded rows: {len(out)}; auto-fixes: {fixlog}')
print(f'Primary Theme: {out["Primary Theme"].value_counts().to_dict()}')


coded rows: 200; auto-fixes: {'q4_consistency': 1, 'q21_parent_yes': 0, 'q11_required_missing': 0}
Primary Theme: {'Male Victimhood': 37, 'Unclear': 31, 'Gender Grievance': 25, 'Self-Discipline': 22, 'Male Accountability': 19, 'Faith and Moral Repair': 14, 'Authority and Submission': 13, 'Egalitarian Partnership': 13, 'Provider and Status': 8, 'Sexual Morality': 6, 'Gender-Based Violence and Consent': 6, 'Marriage and Family': 4, 'Relationship Tactics': 2}


## 5 — Append as new sheet to the existing xlsx (preserve Nigeria sheet)

In [6]:
COUNTRY = 'Kenya'
SHEET_NAME_OUTPUT = 'Kenya - LLM Coding'
SHEET_HEADER_COLOR = 'C65911'
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

export_cols = [c for c in out.columns if c not in ('creator', 'orientation')]
out_export = out[export_cols].copy()

# load existing or create new
if OUT_XLSX.exists():
    wb = openpyxl.load_workbook(OUT_XLSX)
else:
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

# the sheet name to use depends on country (set per-notebook)
SHEET_NAME = globals().get('SHEET_NAME_OUTPUT', 'LLM Coding')
if SHEET_NAME in wb.sheetnames:
    del wb[SHEET_NAME]
ws = wb.create_sheet(SHEET_NAME)
ws.append(list(out_export.columns))
for _, row in out_export.iterrows():
    ws.append([row[c] for c in out_export.columns])

# style
header_color = globals().get('SHEET_HEADER_COLOR', '305496')
header_fill = PatternFill('solid', fgColor=header_color)
header_font = Font(bold=True, color='FFFFFF', size=10)
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(wrap_text=True, vertical='center', horizontal='left')
ws.row_dimensions[1].height = 60
ws.freeze_panes = 'E2'
for col_idx, col_name in enumerate(out_export.columns, 1):
    letter = get_column_letter(col_idx)
    if col_name == 'Comment Text':
        ws.column_dimensions[letter].width = 60
    elif col_name in ('Commenter Post URL', "Influencer's OG Post URL"):
        ws.column_dimensions[letter].width = 35
    elif col_name in ('Primary Theme', 'Secondary Theme 1', 'Secondary Theme 2', 'Target of Claim'):
        ws.column_dimensions[letter].width = 28
    elif col_name.startswith('Q') and ('a' in col_name.split('.')[0].lower()):
        ws.column_dimensions[letter].width = 35
    else:
        ws.column_dimensions[letter].width = 18
for row in ws.iter_rows(min_row=2):
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical='top')
        c.font = Font(size=10)

# methodology sheet
if 'Methodology' in wb.sheetnames: del wb['Methodology']
mws = wb.create_sheet('Methodology')
mws.append(['country', 'metric', 'value'])
country = globals().get('COUNTRY', 'Nigeria')
for row_data in [
    (country, 'Total rows', len(out_export)),
    (country, 'Per creator', N_PER_CREATOR),
    ('Both', 'Model', MODEL),
    ('Both', 'Seed', SEED),
    ('Both', 'Themes vocabulary', ', '.join(THEMES)),
    ('Both', 'Sentiment values', ', '.join(SENTIMENTS)),
    ('Both', 'Emotion values', ', '.join(EMOTIONS)),
    ('Both', 'Tone values', ', '.join(TONES)),
    ('Both', 'Normative Orientation values', ', '.join(NORMATIVE_ORIENTATIONS)),
    ('Both', 'Target of Claim values', ', '.join(TARGETS)),
]:
    mws.append([str(x) for x in row_data])
mws.column_dimensions['A'].width = 12
mws.column_dimensions['B'].width = 28
mws.column_dimensions['C'].width = 80
for cell in mws[1]: cell.font = Font(bold=True)

# preserve sheet order if both exist
order_pref = ['Nigeria - LLM Coding', 'Kenya - LLM Coding', 'Methodology']
ordered = [wb[n] for n in order_pref if n in wb.sheetnames]
wb._sheets = ordered + [s for s in wb._sheets if s not in ordered]

wb.save(OUT_XLSX)
print(f'wrote {OUT_XLSX} ({OUT_XLSX.stat().st_size:,} bytes)')
print(f'sheets: {[s.title for s in wb.worksheets]}')


wrote /Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project/Codebooks/LLM Codebook/LLM Coding - Audience Analysis.xlsx (162,427 bytes)
sheets: ['Nigeria - LLM Coding', 'Kenya - LLM Coding', 'Methodology']


## Notes

- Same prompt as Nigeria — the 13-theme codebook is country-cross-cutting and was validated against Kenya data via 20 keyword probes (Polygamy, Sheng/Swahili, Tribalism, Politics, FGM, Fitness/Hustle all fired <1%; no new themes warranted).
- Cache is keyed by SHA-1 of comment text. Re-running the notebook is free.
- Sample is 200 comments — 50 per creator, balanced 50/50 progressive (Rixpoet + Eddy) vs regressive (Andrew Kibe + Amerix). Maximum is 70 per creator (Andrew has the smallest pool at 70 rows).
- For Kenya rows, the source xlsx provides only one URL per comment. We use it for both 'Commenter Post URL' and 'Influencer's OG Post URL' in the output (Nigeria X data has both URLs combined into one string and we split them).
- Output appends as new sheet 'Kenya - LLM Coding' alongside 'Nigeria - LLM Coding' in the same xlsx.